# Task 2b: Specialist Autoencoders Validation & Hard-Routing Comparison

Evaluates each specialist DAE and the full hard-routing pipeline in:
- **Oracle routing** – known corruption label selects the expert
- **Predicted routing** – classifier prediction selects the expert

Required by Plan 6 §2b.5.


In [ ]:
# 1. Imports
import sys
from pathlib import Path
from collections import defaultdict
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from src.device_utils import get_device
from src.models_dae import ConvDAE
from src.models_classifier import CorruptionClassifier
from src.checkpoint_utils import load_checkpoint
from src.evaluation import PetBenchmarkDataset, image_quality_metrics

DEVICE = get_device()
CLASS_NAMES = ["clean", "salt", "blur", "occlusion"]


In [ ]:
# 2. Checkpoint paths
SALT_CKPT = RESEARCH_ROOT / "checkpoints" / "task2b_specialist_salt_best.pt"
BLUR_CKPT = RESEARCH_ROOT / "checkpoints" / "task2b_specialist_blur_best.pt"
OCCL_CKPT = RESEARCH_ROOT / "checkpoints" / "task2b_specialist_occlusion_best.pt"
CLS_CKPT  = (RESEARCH_ROOT / "checkpoints" / "task2a_classifier_final_best.pt"
             if (RESEARCH_ROOT / "checkpoints" / "task2a_classifier_final_best.pt").exists()
             else RESEARCH_ROOT / "checkpoints" / "task2_classifier_best.pt")


## 1. Load All Specialists and Classifier

In [ ]:
# 3. Load models
checkpoint_paths = {"salt": SALT_CKPT, "blur": BLUR_CKPT, "occlusion": OCCL_CKPT}
missing = [str(p) for p in [*checkpoint_paths.values(), CLS_CKPT] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Train Tasks 2a and 2b first. Missing: " + ", ".join(missing)
    )

def load_dae(path):
    state = torch.load(path, map_location=DEVICE, weights_only=False)["model_state_dict"]
    base_c = state["enc1.0.weight"].shape[0]
    latent  = state["fc_z.weight"].shape[0]
    m = ConvDAE(base_channels=base_c, latent_dim=latent).to(DEVICE)
    m.load_state_dict(state)
    m.eval()
    return m

specialists = {kind: load_dae(path) for kind, path in checkpoint_paths.items()}

cls_state  = torch.load(CLS_CKPT, map_location=DEVICE, weights_only=False)["model_state_dict"]
base_c_cls = cls_state["conv1.0.weight"].shape[0]
classifier = CorruptionClassifier(base_channels=base_c_cls).to(DEVICE)
classifier.load_state_dict(cls_state)
classifier.eval()
print("All specialists and classifier loaded successfully.")


## 2. Per-Specialist Performance across Benchmark Severities

In [ ]:
# 4. Domain performance by corruption type and severity
benchmark = PetBenchmarkDataset(RESEARCH_ROOT / "test_manifest_official.json")
loader = DataLoader(benchmark, batch_size=16, shuffle=False, num_workers=2)

domain_sums   = defaultdict(lambda: defaultdict(float))
domain_counts = defaultdict(int)
routing_records = []

with torch.inference_mode():
    for corrupted, clean, labels, kinds, severities, names in loader:
        corrupted, clean = corrupted.to(DEVICE), clean.to(DEVICE)

        # Classifier probabilities
        cls_logits = classifier(corrupted)
        cls_probs  = torch.softmax(cls_logits, dim=1)
        cls_preds  = cls_logits.argmax(dim=1)

        for i in range(len(clean)):
            kind     = kinds[i]
            severity = severities[i]
            name     = names[i]
            img_in   = corrupted[i:i+1]
            img_gt   = clean[i:i+1]

            # Oracle routing
            if kind == "clean":
                oracle_out = img_in
            else:
                oracle_out = specialists[kind](img_in).clamp(0, 1)

            # Predicted routing
            pred_idx   = cls_preds[i].item()
            pred_name  = CLASS_NAMES[pred_idx]
            if pred_name == "clean":
                pred_out = img_in
            else:
                expert_key = pred_name if pred_name in specialists else "salt"
                pred_out   = specialists[expert_key](img_in).clamp(0, 1)

            oracle_m  = image_quality_metrics(oracle_out, img_gt)
            pred_m    = image_quality_metrics(pred_out,   img_gt)

            key = (kind, severity)
            domain_counts[key] += 1
            for m, v in oracle_m.items():
                domain_sums[(key, "oracle")][m] += v
            for m, v in pred_m.items():
                domain_sums[(key, "predicted")][m] += v

            routing_records.append({
                "name":             name,
                "true":             kind,
                "severity":         severity,
                "predicted_name":   pred_name,
                "correct_route":    (pred_name == kind or (kind == "clean" and pred_name == "clean")),
                "input":            corrupted[i].cpu(),
                "oracle":           oracle_out.squeeze(0).cpu(),
                "predicted_output": pred_out.squeeze(0).cpu(),
                "target":           clean[i].cpu(),
                "oracle_psnr":      oracle_m["psnr"],
                "pred_psnr":        pred_m["psnr"],
                "cls_probs":        cls_probs[i].cpu().numpy(),
            })

# Build comparison table
rows = []
for (kind, severity), n in sorted(domain_counts.items()):
    for routing in ["oracle", "predicted"]:
        row = {"corruption": kind, "severity": severity, "routing": routing, "n": n}
        for m, total in domain_sums[((kind, severity), routing)].items():
            row[m] = round(total / n, 4)
        rows.append(row)
route_df = pd.DataFrame(rows)
display(route_df)


## 3. Oracle Routing vs Predicted Hard-Routing Comparison

In [ ]:
# 5. Pivot table: PSNR oracle vs predicted by corruption and severity
pivot = (route_df.pivot_table(
    index=["corruption", "severity"],
    columns="routing",
    values="psnr"
))
pivot["psnr_drop_dB"] = (pivot["oracle"] - pivot["predicted"]).round(3)
display(pivot)
print(f"Mean PSNR drop oracle→predicted: {pivot['psnr_drop_dB'].mean():.3f} dB")


## 4. Visual Restorations & Misrouting Failure Cases

In [ ]:
# 6. Show correctly-routed examples and misrouting failures
misroutes = [r for r in routing_records if not r["correct_route"]]
examples  = misroutes[:4]
if not examples:
    print("No predicted-routing errors found on this benchmark manifest.")
else:
    fig, axes = plt.subplots(len(examples), 4, figsize=(14, 3 * len(examples)), squeeze=False)
    for row, record in zip(axes, examples):
        images = [record["input"], record["oracle"],
                  record["predicted_output"], record["target"]]
        titles = ["Input",
                  f"Oracle ({record['true']})",
                  f"Predicted ({record['predicted_name']})",
                  "Target"]
        for ax, image, title in zip(row, images, titles):
            ax.imshow(image.permute(1, 2, 0).numpy().clip(0, 1))
            ax.set_title(title, fontsize=8)
            ax.axis("off")
        print(f"{record['name']}: true={record['true']}, predicted={record['predicted_name']}, "
              f"PSNR oracle={record['oracle_psnr']:.2f} dB, predicted={record['pred_psnr']:.2f} dB")
    plt.tight_layout()
    plt.show()
